In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import compute_sample_weight

In [5]:
import warnings

warnings.filterwarnings("ignore")

In [6]:
%load_ext autoreload
%autoreload 2

In [7]:
from ibl_info.decoder_pid import run_decoder_bootstrapping

In [ ]:
# # Generate Synthetic Data
# m_neurons = 500
# n_trials = 200

# neural_data = np.random.randn(m_neurons, n_trials)
# labels = np.random.randint(0, 2, size=(1, n_trials))

# # Add Signal
# signal_mask = labels.flatten() == 1
# neural_data[:50, signal_mask] += 0.5

# # Masks
# cong_mask = np.zeros(n_trials, dtype=bool)
# cong_mask[:100] = True
# incong_mask = ~cong_mask

# # Run Pipeline with 5-Fold CV
# D = 50
# bootstrap_results = run_decoder_bootstrapping(
#     neural_data,
#     labels,
#     subset_size_D=D,
#     n_bootstraps=5,
#     n_splits=5,
#     congruent_mask=cong_mask,
#     incongruent_mask=incong_mask,
#     scale=True,
# )

# first_run = bootstrap_results[0]


# D = 50
# bootstrap_results_unscaled = run_decoder_bootstrapping(
#     neural_data,
#     labels,
#     subset_size_D=D,
#     n_bootstraps=5,  
#     n_splits=5,  
#     congruent_mask=cong_mask,
#     incongruent_mask=incong_mask,
#     scale=False,
# )

# first_run_unscaled = bootstrap_results_unscaled[0]

In [3]:
# try with real data

In [8]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from iblatlas.atlas import BrainRegions
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from brainbox.task.trials import get_event_aligned_raster, get_psth
from iblatlas.atlas import AllenAtlas
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from brainbox.population.decode import get_spike_counts_in_bins
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox import singlecell
from tqdm.notebook import tqdm
import seaborn as sns

from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units

In [5]:
one = ONE()
subject_id = "CSH_ZAD_022"
eid = "a82800ce-f4e3-4464-9b80-4c3d6fade333"
session_id = eid

session_id = eid
pids, probes = one.eid2pid(session_id)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

In [7]:
from ibl_info.prepare_data_pid import get_new_cinc_intervals


trials, mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)
trials = trials[mask]


intervals, target_variable, congruent_flags, incongruent_flags = get_new_cinc_intervals(
    trials, "stim"
)

In [8]:
from ibl_info.prepare_data_pid import prepare_ephys_data
from ibl_info.utils import check_config

config = check_config()

In [9]:
np.unique(clusters["acronym"])

array(['APN', 'CA1', 'CA3', 'COApm', 'DG-mo', 'HPF', 'LGd-co', 'LGd-ip',
       'LGd-sh', 'PA', 'TH', 'VISa1', 'VISa2/3', 'VISa4', 'VISa5',
       'VISam5', 'VISam6a', 'ZI', 'alv', 'ar', 'cpd', 'root'],
      dtype=object)

In [23]:
binned_spikes, actual_regions, n_units, cluster_uuids_list = prepare_ephys_data(
    spikes, clusters, intervals, ["LGd"], minimum_units=config["min_units_decoding"]
)

Region found LGd, 67


In [ ]:
from ibl_info.decoder_pid import compute_decoder_pid


spike_data = binned_spikes[0]  # trials x neurons, what we need
trial_count = np.zeros((3))

trial_count[0] = intervals.shape[0]
trial_count[1] = np.sum(congruent_flags)
trial_count[2] = np.sum(incongruent_flags)

# now we add the new decoder functions
information_pickle = {}
information_pickle["neurons"] = spike_data.shape[0]
information_pickle["trials"] = trial_count


In [31]:
import warnings

warnings.filterwarnings("ignore")

In [65]:
information_results, results = compute_decoder_pid(
    target=target_variable,
    spikes=spike_data,
    n_bootstaps=config["n_bootstraps_decoding"],
    n_bins=config["n_bins_decoding"],
    congruent_mask=congruent_flags,
    incongruent_mask=incongruent_flags,
)

Starting bootstrapping: 50 iterations with 5-Fold CV...
Bootstrapping complete.


In [66]:
information_results.shape

(50, 2, 7)

In [67]:
mean_results = np.mean(information_results, axis=0)

In [68]:
information_results.shape

(50, 2, 7)

In [69]:
np.mean(information_results[:, 0, 2] - information_results[:, 0, 0] - information_results[:, 0, 1])

np.float64(-0.09282929991418182)

In [70]:
np.mean(information_results[:, 1, 2] - information_results[:, 1, 0] - information_results[:, 1, 1])

np.float64(-0.07561365665733123)

In [10]:
one = ONE(mode="local")
subject_id = "CSH_ZAD_022"
eid = "a82800ce-f4e3-4464-9b80-4c3d6fade333"
session_id = eid

In [11]:
session_id = eid
pids, probes = one.eid2pid(session_id)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

NotImplementedError: Converting to probe ID requires remote connection

NameError: name 'one' is not defined